# 01. Khám phá dữ liệu thu âm

Notebook này chỉ đọc dữ liệu, không sửa gì. Phần xử lý nằm trong `ml/` (cùng hàm với lệnh `python -m ml.check_data`), ở đây mình dùng để xem, nghe và hiểu dữ liệu trước khi train.

Các phần:
1. Số mẫu theo phiên và nhãn
2. Các file nên nghe lại (nói nhỏ, bị cắt, silence có tiếng...), nghe được ngay trong notebook
3. Dạng sóng và phổ tần của mỗi nhãn
4. Thời điểm bắt đầu nói (để xem có nên dùng `--align` không)
5. PCA của đặc trưng MFCC: các nhãn có tách nhau không

In [ ]:
# Mở notebook từ thư mục notebooks/ hay thư mục gốc đều chạy được
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Audio, Image, Markdown, display
pd.set_option('display.max_colwidth', 80)
print('Thư mục làm việc:', ROOT)

In [ ]:
# Tham số
MANIFEST = os.environ.get('VOVC_MANIFEST', 'data/manifest.csv')
MAX_PLAY = 6        # số file tối đa phát trong mỗi ô nghe

## 1. Số mẫu theo phiên

In [ ]:
from app.common import LABELS, COMMANDS
from ml.dataset import read_manifest, FIELDS

rows = read_manifest(MANIFEST, require_splits=False)
df = pd.DataFrame([{**{k: r[k] for k in FIELDS}, 'file': r['_file']} for r in rows])
counts = pd.crosstab([df.speaker_id, df.session_id], df.label).reindex(columns=LABELS, fill_value=0)
print(f'{len(df)} bản thu, {df.speaker_id.nunique()} người nói, {counts.shape[0]} phiên')
counts

In [ ]:
missing = counts[(counts == 0).any(axis=1)]
if len(missing):
    display(Markdown('**Phiên thiếu nhãn** (mỗi phiên dùng cho validation/test cần đủ 6 nhãn):'))
    display(missing)
else:
    print('Mọi phiên đều đủ 6 nhãn.')

## 2. Kiểm tra chất lượng từng file

In [ ]:
from ml.check_data import check
from ml.quality import describe

results, _, _, _ = check(MANIFEST)
quality = pd.DataFrame([{
    'path': r['row']['path'], 'label': r['row']['label'], 'session': r['row']['session_id'],
    'peak': round(r['stats']['peak'], 3), 'speech_start': r['stats']['speech_start'],
    'speech_end': r['stats']['speech_end'], 'issues': describe(r['issues'])} for r in results])
flagged = [r for r in results if r['issues']]
print(f'{len(flagged)}/{len(results)} file nên nghe lại')
quality[quality.issues != '']

In [ ]:
# Nghe các file bị gắn cờ. Luật chỉ là ước lượng, nghe bằng tai mới chắc.
for r in flagged[:MAX_PLAY]:
    display(Markdown(f"`{r['row']['path']}`, **{r['row']['label']}**: {describe(r['issues'])}"))
    display(Audio(filename=r['row']['_file']))

In [ ]:
# Nghe một mẫu bình thường của mỗi nhãn để so sánh
for label in LABELS:
    ok = [r for r in results if r['row']['label'] == label and not r['issues']]
    if ok:
        display(Markdown(f"**{label}** {COMMANDS.get(label, '')}"))
        display(Audio(filename=ok[0]['row']['_file']))

## 3. Dạng sóng và phổ tần

Vùng màu vàng là đoạn mà `ml.quality.analyze` cho là có tiếng nói. Nhìn phổ tần thì "bật" và "tắt" khác nhau chủ yếu ở phụ âm đầu (b và t), còn "đèn" và "quạt" khác nhau ở nửa sau.

In [ ]:
from ml import viz

viz.examples(rows)
plt.show()

## 4. Bắt đầu nói lúc nào

Mỗi vạch là đoạn có tiếng nói trong một bản thu. Nếu các vạch lệch nhau nhiều thì 4 đoạn thời gian của đặc trưng sẽ rơi vào những phần khác nhau của cùng một câu, lúc đó nên thử `--align`. Vạch nào chạm đường đỏ là có thể bị cắt chữ cuối.

In [ ]:
timing = [{'label': r['row']['label'], **r['stats']} for r in results]
viz.speech_timing(timing)
plt.show()

# Trung vị theo nhãn (bỏ silence: không có tiếng nói nên cột thời điểm vô nghĩa)
speech_only = pd.DataFrame([t for t in timing if t['label'] != 'silence'])
speech_only.groupby('label')[['peak', 'speech_start', 'speech_end', 'active_s']].median().round(2).reindex([l for l in LABELS if l != 'silence'])

## 5. PCA của đặc trưng

Mỗi chấm là một bản thu, vector MFCC 104 chiều được chiếu xuống 2 chiều. Nhãn nào tách thành cụm riêng thì dễ phân loại, hai nhãn chồng lên nhau thì model dễ nhầm. PCA chỉ để nhìn sơ bộ, model thật dùng đủ 104 chiều.

In [ ]:
from ml.features import FeatureConfig

viz.feature_map(rows)
plt.show()
viz.feature_map(rows, FeatureConfig(align=True))
plt.show()

## Làm tiếp

- Nghe lại các file bị gắn cờ. File hỏng thì xóa cả file WAV lẫn dòng trong `data/manifest.csv`, hoặc chạy `python -m ml.check_data --drop-flagged data/manifest_clean.csv`.
- Có ít nhất 3 phiên đủ 6 nhãn (ví dụ s01, s02, s03) thì chuyển sang notebook 02.
- Trước khi commit nhớ Clear All Outputs, vì output có âm thanh giọng nói.